# Week 1 Day 2 Lab - Introducing UnderCover

## LLMs and Agents on a commercial task: insurance claims

A claims handler at UnderCover Travel Insurance has already decided each of the 25 claims in its queue. We send every claim to a few models, along with the customer's policy record and the policy wording. Each model replies with a decision of approve, decline or refer, a rationale and a payout. Then we compare the answers with the handlers' decisions and look at what each model cost.

At the end we give the same job to a LangChain agent that uses tools to look up the policy and record its decision.

## 4 parts:

PART 1: Examine the data

PART 2: Run with some models

PART 3: Optional: larger models and Jev

PART 4: An Agent via LangChain

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This lab calls paid models through OpenRouter and costs only a few cents. If you're OK to spend that, you'll need some credit on your account at <a href="https://openrouter.ai">openrouter.ai</a>. If you'd rather not spend anything, there's a free version of this lab in <a href="../../week1day2free.ipynb">week1day2free.ipynb</a>.<br/><br/>
            Please execute each cell and check that you get similar results to mine (the models vary a little from run to run). Add print statements and try small adjustments to make the code your own.<br/><br/>
            Some cells have code that isn't about AI, like the charts. It's not important to understand those in detail.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/resources.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Resources</h2>
            <span style="color:#118AB2;">The <a href="../guides/01_intro.ipynb">guides</a> folder in this repo has more information on topics like the command line, Git, Python and debugging.<br/><br/>
            If this lab is too much, consider courses 1, 2, 4 and 5 in <a href="https://edwarddonner.com/curriculum">my curriculum</a>: AI Builder, AI Coder, and the Core and Agentic Tracks of AI Engineer.<br/><br/>
            If it's too slow, put me on 2x and zip through it! There's some interesting content on Jev for the pros.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/connect.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Stay in touch</h2>
            <span style="color:#118AB2;">Please message me in Udemy anytime with questions, or email me at <a href="mailto:ed@edwarddonner.com">ed@edwarddonner.com</a>.<br/><br/>
            Let's connect on <a href="https://www.linkedin.com/in/eddonner/">LinkedIn</a>! And subscribe to my <a href="https://www.youtube.com/@edward.donner">YouTube channel</a> for more on LLMs, agents and coding agents.
            </span>
        </td>
    </tr>
</table>

In [ ]:
import json
import os
import time
from pathlib import Path
from typing import Literal
from collections import defaultdict

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openai import OpenAI
from openrouter import OpenRouter
from pydantic import BaseModel, Field
import time

## Connect to OpenRouter - and please read note here

The OpenRouter key comes from the `.env` file you made on day 1 in the `undercover` folder.

### NOTE:

If this is a new OpenRouter account, then you might get an error at some point from OpenRouter like: "new accounts are limited to 20 requests per minute for this model"

If you get that error, then add this line immediately after calling an LLM:

`time.sleep(3)`

and you should be in good shape.

In [ ]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

## Part 1: Load and examine the data

The seed data is in the repo's `data` folder. `get_sample_data` returns the claims and the handlers' decisions on them, keyed by claim id. The model is sent the claims, the policies and the policy wording.

By default we work with the queue of 25 claims. If you are using free models with a daily limit, set `TINY = True` to keep 5 of them.

In [ ]:
DATA = Path("../../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

In [ ]:
TINY = False

claims, labels = get_sample_data(tiny=TINY)
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")
print(f"The policy wording is {len(policy_wording.split()):,} words")

## The prompt

The system prompt has the instructions and the whole policy wording. The user prompt has the customer's policy record and the claim, both as JSON.

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""

SYSTEM_PROMPT = f"""{INSTRUCTIONS}
{policy_wording}
"""


def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""


def make_messages(claim):
    """Build the messages for one claim: the task and the wording, then the policy record and the claim."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(claim)},
    ]

## Part 4: Using LangChain create_agent

Our approaches so far have used a single LLM call with Structured Outputs.

Now we'll use a different approach: an AI agent.

An LLM, with tools, in a loop, to achieve a goal!

We'll use LangChain's `create_agent` to run the loop for us.

We give it 2 tools. `get_policy` looks up the customer's policy record. `record_decision` saves the decision.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Reminder: LangChain create_agent</h2>
            <span style="color:#118AB2;">1. Describe the LLM with a LangChain class like <code>ChatOpenAI</code><br/>
            2. Turn a function into a tool with <code>@tool</code><br/>
            3. <code>create_agent</code> builds the agent and <code>invoke()</code> runs its loop<br/><br/>
            Behind the scenes, a LangGraph graph is created; we'll find out more in Week 2.
            </span>
        </td>
    </tr>
</table>

In [ ]:
llm = ChatOpenAI(model="openrouter/free", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)

messages = [
    {"role": "system", "content": "You are a helpful assistant who answers in the style of a pirate."},
    {"role": "user", "content": "What is the meaning of the universe?"},
]

response = llm.invoke(messages)
display(Markdown(response.content))

In [ ]:
recorded = {}


@tool
def get_policy(policy_id: str) -> str:
    """Look up a customer's policy record: the plan, the add-ons, the purchase date, the trip and the travelers."""
    policy = policies.get(policy_id)
    return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"


@tool
def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"], payout: float, rationale: str) -> str:
    """Record the decision on a claim in the claims system.

    payout is the amount to pay in US dollars, or 0 for a decline or a referral.
    rationale is 2-3 sentences to explain the decision along with the relevant section of the policy.
    """
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"

In [ ]:
AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.

{policy_wording}
"""


def make_claim_prompt(claim):
    """The claim alone, as JSON. The agent looks up the policy record for itself."""
    return f"""# Claim

{json.dumps(claim, indent=2)}
"""


Now the whole queue. Each claim takes three model calls: one to ask for the policy, one to record the decision and one to finish. OpenRouter's cost for each call is in the message metadata. A claim with no recorded decision counts as wrong.

In [ ]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1

def decide_with_agent(claim, model, effort):
    """Run the agent on one claim and check what it recorded against the handler's decision."""
    start = time.time()
    llm = ChatOpenAI(model=model, base_url=OPENROUTER_BASE_URL, reasoning_effort=effort, api_key=openrouter_api_key)
    agent = create_agent(model=llm, tools=[get_policy, record_decision], system_prompt=AGENT_SYSTEM_PROMPT)
    result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claim)}]})
    answer = recorded.get(claim["claim_id"], {"decision": "nothing recorded", "payout": 0.0, "rationale": ""})
    label = labels[claim["claim_id"]]
    cost = sum(message.response_metadata["token_usage"]["cost"] for message in result["messages"] if message.type == "ai")
    return {
        "model": model,
        "effort": effort,
        "technique": "Agent",
        "claim_id": claim["claim_id"],
        **answer,
        "difficulty": label["difficulty"],
        "right": is_right(answer, label),
        "cost": cost,
        "seconds": round(time.time() - start, 1),
    }



In [ ]:
models = [("openai/gpt-6-luna", "low"),
        ("qwen/qwen3.7-flash", "medium"),
        ("openai/gpt-oss-20b", "low"),
        ("mistralai/mistral-small-2603", None),
        ("deepseek/deepseek-v4.1-flash-1017", "medium"),
        ("inception/mercury-3", None), 
        ("openai/gpt-5-nano", "high"),
        ("inclusionai/ling-3.0-flash", "high"),
        ("xiaomi/mimo-v2.6-flash", "low"),
        ("nvidia/nemotron-3-ultra-550b-a55b", "medium"),
        ("google/gemini-3.8-flash",None)]

recorded.clear()
experiments = []

def run_agents(model, effort): 
    agent_results = []
    print(f"Running model {model} on {effort} effort")
    for claim in claims:
        agent_results.append(decide_with_agent(claim, model, effort))
    print(f"Model {model} completed")
    experiments.append(agent_results)

for model, effort in models:
    run_agents(model=model, effort=effort)

    print(f"{len(recorded)} of {len(claims)} claims have a recorded decision")

In [ ]:
experiments

## Putting it all together

Here is every experiment in one table. Cost is for the whole queue and speed is the average time for one claim. Jev isn't included because it only makes the decision.

In [ ]:
rows = []
for results in experiments:
    right = sum(result["right"] for result in results)
    cost = sum(result["cost"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    rows.append({
        "LLM": results[0]["model"],
        "reasoning": results[0]["effort"] or "default",
        "technique": results[0]["technique"],
        "accuracy": f"{right} / {len(results)}",
        "cost": f"${cost:.4f}",
        "speed": f"{seconds / len(results):.1f} seconds a claim",
    })

pd.DataFrame(rows)

## Results using different cheaper models in agent mode

<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>LLM</th>
      <th>reasoning</th>
      <th>technique</th>
      <th>accuracy</th>
      <th>cost</th>
      <th>speed</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>openai/gpt-6-luna</td>
      <td>low</td>
      <td>Agent</td>
      <td>18 / 25</td>
      <td>$0.0088</td>
      <td>6.7 seconds a claim</td>
    </tr>
    <tr>
      <th>1</th>
      <td>qwen/qwen3.7-flash</td>
      <td>medium</td>
      <td>Agent</td>
      <td>16 / 25</td>
      <td>$0.0191</td>
      <td>45.5 seconds a claim</td>
    </tr>
    <tr>
      <th>2</th>
      <td>openai/gpt-oss-20b</td>
      <td>low</td>
      <td>Agent</td>
      <td>11 / 25</td>
      <td>$0.0094</td>
      <td>5.4 seconds a claim</td>
    </tr>
    <tr>
      <th>3</th>
      <td>mistralai/mistral-small-2603</td>
      <td>default</td>
      <td>Agent</td>
      <td>8 / 25</td>
      <td>$0.0141</td>
      <td>8.3 seconds a claim</td>
    </tr>
    <tr>
      <th>4</th>
      <td>deepseek/deepseek-v4.1-flash-1017</td>
      <td>medium</td>
      <td>Agent</td>
      <td>17 / 25</td>
      <td>$0.0243</td>
      <td>8.7 seconds a claim</td>
    </tr>
    <tr>
      <th>5</th>
      <td>inception/mercury-3</td>
      <td>default</td>
      <td>Agent</td>
      <td>17 / 25</td>
      <td>$0.0214</td>
      <td>4.9 seconds a claim</td>
    </tr>
    <tr>
      <th>6</th>
      <td>openai/gpt-5-nano</td>
      <td>high</td>
      <td>Agent</td>
      <td>12 / 25</td>
      <td>$0.1233</td>
      <td>171.1 seconds a claim</td>
    </tr>
    <tr>
      <th>7</th>
      <td>inclusionai/ling-3.0-flash</td>
      <td>high</td>
      <td>Agent</td>
      <td>15 / 25</td>
      <td>$0.0057</td>
      <td>10.3 seconds a claim</td>
    </tr>
    <tr>
      <th>8</th>
      <td>xiaomi/mimo-v2.6-flash</td>
      <td>low</td>
      <td>Agent</td>
      <td>18 / 25</td>
      <td>$0.0235</td>
      <td>43.6 seconds a claim</td>
    </tr>
    <tr>
      <th>9</th>
      <td>nvidia/nemotron-3-ultra-550b-a55b</td>
      <td>medium</td>
      <td>Agent</td>
      <td>16 / 25</td>
      <td>$0.1569</td>
      <td>6.8 seconds a claim</td>
    </tr>
    <tr>
      <th>10</th>
      <td>google/gemini-3.8-flash</td>
      <td>default</td>
      <td>Agent</td>
      <td>19 / 25</td>
      <td>$0.3379</td>
      <td>18.5 seconds a claim</td>
    </tr>
  </tbody>
</table>
</div>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">This lab is about experimenting with different LLMs and Agents to understand the accuracy, cost and speed.<br/><br/>
            Based on our initial experiments, Luna (low) is able to perform well at a low cost. More expensive models can perform better.<br/><br/>
            Challenge: with a goal of at least 15 / 25 correct consistently, using the agent, what is the cheapest model you can find?<br/><br/>
            I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#069D75;">Congratulations! And if I could disturb you with a quick request:</h2>
            <span style="color:#069D75;">You've finished the first main lab. You compared several models and an agent on UnderCover's claims, with accuracy, cost and speed side by side.<br/><br/>
            On Day 3 we bring in coding agents and start building the UnderCover platform.<br/><br/>
            <b>I have a quick request for you. If you have a moment to rate the course, I'd hugely appreciate it. It's one of the main ways Udemy decides whether
            to show the course to others, and I'd be super grateful.</b>
            </span>
        </td>
    </tr>
</table>